In [1]:
import psycopg2
from dotenv import load_dotenv
import os 
load_dotenv()
url = os.getenv("DATABASE_URL")

In [2]:
conn = psycopg2.connect(url, sslmode="require")
cur = conn.cursor()

In [3]:
cur.execute(
    """
    SELECT DISTINCT ticker
    FROM orderbooks
    """
)
tickers = [row[0] for row in cur.fetchall()]
print(tickers)

['SR320CC6', 'SR320CD6A', 'SR320CO6D', 'SR310CD6', 'SR300CD6A', 'SR310CC6D', 'SR300CP6', 'SR320CP6A', 'SR320CD6', 'SR300CO6D', 'SR320CP6', 'SR320CC6D', 'SR310CP6', 'SR300CD6', 'SR310CO6D', 'SR310CP6A', 'SR310CC6', 'SR300CC6D', 'SR300CP6A', 'SR300CC6', 'SR310CD6A']


In [4]:
import pandas as pd

ticker = "SR320CC6"
columns = ["id", "ticker", "timestamp", "bids", "asks"] 

query = f"""
SELECT {', '.join(columns)}
FROM orderbooks
WHERE ticker = '{ticker}'
"""

option_df = pd.read_sql_query(query, conn)

option_df['timestamp'] = pd.to_datetime(option_df['timestamp'])
option_df.set_index('timestamp', inplace=True)

option_df['best_bid'] = option_df['bids'].apply(lambda bids: bids[0]['price'] if bids else None)
option_df['best_ask'] = option_df['asks'].apply(lambda asks: asks[0]['price'] if asks else None)
option_df['mid_price'] = option_df['best_bid'] + (option_df['best_ask'] - option_df['best_bid'])/2
option_df['spread'] = option_df['best_ask'] - option_df['best_bid']

option_df.drop(columns=['bids', 'asks', 'id'], inplace=True)
option_df = option_df[~option_df.index.duplicated(keep='first')]
option_df.drop_duplicates(inplace=True)
option_df.head()

C:\Users\bakae\AppData\Local\Temp\ipykernel_23340\92435234.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  option_df = pd.read_sql_query(query, conn)


,ticker,best_bid,best_ask,mid_price,spread
timestamp,,,,,
2026-03-11 16:44:09+00:00,SR320CC6,1.11,1.50,1.305,0.39
2026-03-12 07:27:03+00:00,SR320CC6,0.70,1.46,1.080,0.76
2026-03-12 07:27:06+00:00,SR320CC6,0.93,1.46,1.195,0.53
2026-03-11 16:50:00+00:00,SR320CC6,1.11,1.49,1.300,0.38
2026-03-11 17:12:59+00:00,SR320CC6,1.30,1.49,1.395,0.19


In [5]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from src.mm_backtest import run_backtest

backtest_results = run_backtest(option_df, inventory_k=1, order_size=10, inventory_limit=1000)

{'pnl_series': [np.float64(0.0), np.float64(-0.3000000000000007), np.float64(0.8499999999999996), np.float64(1.8999999999999986), np.float64(2.849999999999998), np.float64(2.8999999999999986), np.float64(1.8499999999999979), np.float64(1.799999999999999), np.float64(1.6999999999999993), np.float64(1.5999999999999979), np.float64(1.6499999999999986), np.float64(1.3499999999999996), np.float64(1.75), np.float64(-6.500000000000001), np.float64(2.0), np.float64(1.8499999999999979), np.float64(1.4999999999999982), np.float64(1.25), np.float64(1.0999999999999979), np.float64(1.1999999999999993), np.float64(1.3999999999999986), np.float64(2.549999999999999), np.float64(1.9499999999999993), np.float64(2.0), np.float64(2.349999999999998), np.float64(2.299999999999997), np.float64(2.3999999999999986), np.float64(2.2499999999999982), np.float64(-0.45000000000000284), np.float64(nan), np.float64(2.1999999999999993), np.float64(2.1499999999999986), np.float64(2.9999999999999982), np.float64(3.94999